In [ ]:
# !pip install xgboost

In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import GridSearchCV

# Load the training data (replace 'train.csv' with your actual file path)
train_df = pd.read_csv('train.csv', index_col='id')

# Encode 'Sex' column (1 for male, 2 for female)
train_df['Sex'] = train_df['Sex'].map({'male': 1, 'female': 2})

# Ensure numeric columns
numeric_cols = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Calories']
train_df[numeric_cols] = train_df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Handle missing values (use training set means for consistency)
train_means = train_df.mean()
train_df = train_df.fillna(train_means)

# Feature engineering
train_df['BMI'] = train_df['Weight'] / (train_df['Height'] / 100) ** 2
train_df['Duration_Heart_Rate'] = train_df['Duration'] * train_df['Heart_Rate']
train_df['Duration_Squared'] = train_df['Duration'] ** 2
train_df['Heart_Rate_Squared'] = train_df['Heart_Rate'] ** 2
train_df['Age_BMI'] = train_df['Age'] * train_df['BMI']
train_df['log_Duration'] = np.log1p(train_df['Duration'])
train_df['Heart_Rate_Body_Temp'] = train_df['Heart_Rate'] * train_df['Body_Temp']

# Transform target to log(Calories + 1) for RMSLE
train_df['log_Calories'] = np.log1p(train_df['Calories'])

# Features and target
features = ['Sex', 'Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 
            'BMI', 'Duration_Heart_Rate', 'Duration_Squared', 'Heart_Rate_Squared', 
            'Age_BMI', 'log_Duration', 'Heart_Rate_Body_Temp']
X_train = train_df[features]
y_train = train_df['log_Calories']

# Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [4, 6],
    'learning_rate': [0.01, 0.05],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}
model = XGBRegressor(random_state=42)
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='neg_mean_squared_log_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best model
model = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation RMSLE: {np.sqrt(-grid_search.best_score_):.4f}")

# Evaluate on training data
y_train_pred_log = model.predict(X_train)
y_train_pred = np.expm1(y_train_pred_log)
train_rmsle = np.sqrt(mean_squared_log_error(train_df['Calories'], np.maximum(y_train_pred, 0)))
print(f"Training RMSLE: {train_rmsle:.4f}")

# Load and preprocess test data (replace 'test.csv' with your actual file path)
test_df = pd.read_csv('test.csv', index_col='id')
test_df['Sex'] = test_df['Sex'].map({'male': 1, 'female': 2})
test_df[numeric_cols[:-1]] = test_df[numeric_cols[:-1]].apply(pd.to_numeric, errors='coerce')
test_df = test_df.fillna(train_means)  # Use training set means

# Feature engineering for test data
test_df['BMI'] = test_df['Weight'] / (test_df['Height'] / 100) ** 2
test_df['Duration_Heart_Rate'] = test_df['Duration'] * test_df['Heart_Rate']
test_df['Duration_Squared'] = test_df['Duration'] ** 2
test_df['Heart_Rate_Squared'] = test_df['Heart_Rate'] ** 2
test_df['Age_BMI'] = test_df['Age'] * test_df['BMI']
test_df['log_Duration'] = np.log1p(test_df['Duration'])
test_df['Heart_Rate_Body_Temp'] = test_df['Heart_Rate'] * test_df['Body_Temp']

# Features for prediction
X_test = test_df[features]

# Make predictions and clip to training Calories range
predictions_log = model.predict(X_test)
predictions = np.expm1(predictions_log)
calories_min, calories_max = train_df['Calories'].min(), train_df['Calories'].max()
predictions = np.clip(predictions, calories_min, calories_max)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df.index,
    'Calories': predictions.round(3)
})

# Save to CSV
submission_df.to_csv('submission_xgb_tuned.csv', index=False)

print("Submission file 'submission_xgb_tuned.csv' created successfully!")

Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.8}
Best Cross-Validation RMSLE: 0.0173
Training RMSLE: 0.0581
Submission file 'submission_xgb_tuned.csv' created successfully!


In [3]:
print(f"test_df: {test_df.describe()}")

test_df:                  Sex            Age         Height         Weight  \
count  250000.000000  250000.000000  250000.000000  250000.000000   
mean        1.501124      41.452464     174.725624      75.147712   
std         0.500000      15.177769      12.822039      13.979513   
min         1.000000      20.000000     127.000000      39.000000   
25%         1.000000      28.000000     164.000000      63.000000   
50%         2.000000      40.000000     174.000000      74.000000   
75%         2.000000      52.000000     185.000000      87.000000   
max         2.000000      79.000000     219.000000     126.000000   

            Duration     Heart_Rate      Body_Temp            BMI  \
count  250000.000000  250000.000000  250000.000000  250000.000000   
mean       15.415428      95.479084      40.036093      24.367731   
std         8.349133       9.450161       0.778448       1.511764   
min         1.000000      67.000000      37.100000      13.850416   
25%         8.000000    

In [4]:
importances = pd.Series(model.feature_importances_, index=features)
print(importances.sort_values(ascending=False))

Duration_Squared        0.414815
Duration_Heart_Rate     0.365763
Duration                0.188456
Heart_Rate_Body_Temp    0.015355
Heart_Rate_Squared      0.004235
Age                     0.003902
Sex                     0.002887
Age_BMI                 0.002546
Weight                  0.000765
Heart_Rate              0.000732
Height                  0.000220
Body_Temp               0.000161
log_Duration            0.000114
BMI                     0.000048
dtype: float32


In [5]:
from sklearn.model_selection import cross_val_score

In [6]:
from lightgbm import LGBMRegressor
model = LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_log_error')
print(f"LightGBM Cross-Validation RMSLE: {np.sqrt(-scores.mean()):.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.049620 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1502
[LightGBM] [Info] Number of data points in the train set: 750000, number of used features: 14
[LightGBM] [Info] Start training from score 4.141144
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016413 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1503
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 14
[LightGBM] [Info] Start training from score 4.141530
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.061361 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1506
[LightGBM] [Info] Number of data points in the train s

In [7]:
from sklearn.ensemble import RandomForestRegressor

In [8]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions_log = rf_model.predict(X_test)
rf_predictions = np.expm1(rf_predictions_log)
ensemble_predictions = 0.6 * predictions + 0.4 * rf_predictions  # Weighted average
submission_df = pd.DataFrame({'id': test_df.index, 'Calories': ensemble_predictions.round(3)})
submission_df.to_csv('submission_ensemble.csv', index=False)